In [ ]:

import os
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.font_manager import FontProperties, fontManager

# Figure 5: Example agroclimatic profiles

# Repository root
ROOT_DIR = Path.cwd()

if ROOT_DIR.name == "figure_scripts":
    ROOT_DIR = ROOT_DIR.parent

# Input data
DATA_FILE = (
    ROOT_DIR
    / "dataset"
    / "data"
    / "processed"
    / "agroclimate_with_indicators.parquet"
)

# Output directory
OUTPUT_DIR = ROOT_DIR / "figure_scripts" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PNG_FILE = OUTPUT_DIR / "figure5_agroclimatic_profiles.png"
PDF_FILE = OUTPUT_DIR / "figure5_agroclimatic_profiles.pdf"

# Configure font
FONT_FILE = ROOT_DIR / "calibri.ttf"

if FONT_FILE.exists():
    fontManager.addfont(str(FONT_FILE))
    calibri = FontProperties(fname=str(FONT_FILE))
    plt.rcParams["font.family"] = calibri.get_name()
else:
    calibri = FontProperties(family="DejaVu Sans")
    plt.rcParams["font.family"] = "DejaVu Sans"

plt.rcParams["font.size"] = 13

# Load required columns
data = pd.read_parquet(
    DATA_FILE,
    columns=[
        "grid_id",
        "latitude",
        "longitude",
        "date",
        "water_balance"
    ]
)

data["date"] = pd.to_datetime(data["date"])

# Calculate long-term mean water balance for each grid
grid_summary = (
    data
    .groupby(
        ["grid_id", "latitude", "longitude"],
        as_index=False
    )
    .agg(
        mean_water_balance=(
            "water_balance",
            "mean"
        )
    )
)

# Select grids based on water-balance percentiles
percentiles = [0.05, 0.25, 0.50, 0.75, 0.95]

selected_grids = []

for p in percentiles:
    target = grid_summary[
        "mean_water_balance"
    ].quantile(p)

    idx = (
        (
            grid_summary["mean_water_balance"]
            - target
        )
        .abs()
        .idxmin()
    )

    selected_grids.append(
        grid_summary.loc[idx]
    )

selected_grids = pd.DataFrame(
    selected_grids
)

selected_grids["percentile"] = [
    "P5",
    "P25",
    "P50",
    "P75",
    "P95"
]

print("Selected grids:")
print(
    selected_grids[
        [
            "percentile",
            "grid_id",
            "latitude",
            "longitude",
            "mean_water_balance"
        ]
    ].to_string(index=False)
)

# Calculate monthly water balance
selected_ids = (
    selected_grids["grid_id"]
    .tolist()
)

profile_data = data[
    data["grid_id"].isin(selected_ids)
].copy()

profile_data["month"] = (
    profile_data["date"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

monthly_profile = (
    profile_data
    .groupby(
        ["grid_id", "month"],
        as_index=False
    )
    .agg(
        water_balance=(
            "water_balance",
            "sum"
        )
    )
)

# Add percentile labels
monthly_profile = monthly_profile.merge(
    selected_grids[
        ["grid_id", "percentile"]
    ],
    on="grid_id",
    how="left"
)

# Plot monthly water-balance profiles
fig, ax = plt.subplots(
    figsize=(12, 7)
)

for label in [
    "P5",
    "P25",
    "P50",
    "P75",
    "P95"
]:
    subset = monthly_profile[
        monthly_profile["percentile"] == label
    ]

    ax.plot(
        subset["month"],
        subset["water_balance"],
        linewidth=1.4,
        label=label
    )

# Zero reference line
ax.axhline(
    0,
    linewidth=0.8,
    linestyle="--"
)

ax.set_xlabel(
    "Year",
    fontproperties=calibri,
    fontsize=16
)

ax.set_ylabel(
    "Monthly water balance (mm/month)",
    fontproperties=calibri,
    fontsize=16
)

# Format axis ticks
ax.tick_params(
    axis="both",
    labelsize=16
)

for label in (
    ax.get_xticklabels()
    + ax.get_yticklabels()
):
    label.set_fontproperties(calibri)
    label.set_fontsize(13)

# Panel label
ax.text(
    0.01,
    0.95,
    "(a)",
    transform=ax.transAxes,
    fontproperties=calibri,
    fontsize=16
)

# Legend
ax.legend(
    title="Water-balance percentile",
    prop=calibri,
    fontsize=16,
    title_fontproperties=calibri,
    frameon=True
)

# Grid
ax.grid(
    axis="y",
    linestyle="--",
    linewidth=0.4,
    alpha=0.4
)

plt.tight_layout()

# Save figures
fig.savefig(
    PNG_FILE,
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    PDF_FILE,
    format="pdf",
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

print("\nFigure 5 selesai.")
print("PNG:", PNG_FILE)
print("PDF:", PDF_FILE)
